# BoTSORT with default parameters on SoccerNet, DanceTrack, SportsMOT and MOT17

- **SoccerNet**: CMC enabled (video frames available in `soccernet/soccernet_data/tracking/test/`)
- **DanceTrack**: CMC disabled (no video frames available, detection files only)
- **SportsMOT**: CMC enabled (video frames available in `sportsmot/val/` and `sportsmot/test/`)
- **MOT17**: CMC enabled (video frames available in `mot17/test/`), test submission zip (no GT)

In [5]:
import os
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd
import supervision as sv

from trackers.core.botsort import BoTSORTTracker
from trackers.eval import evaluate_mot_sequences

DUMMY_FRAME = np.zeros((100, 100, 3), dtype=np.uint8)


def build_dets_index(det_list):
    dets_by_frame = defaultdict(list)
    for line in det_list:
        frame_id = int(line.split(",")[0])
        dets_by_frame[frame_id].append(line)
    return dets_by_frame

## SoccerNet (test set) — CMC enabled

In [2]:
SOCCERNET_TEST_DET_ROOT = os.path.join("soccernet", "SoccerNet_dets", "SoccerNet_tracking_2022_all_dets")
SOCCERNET_TEST_GT_ROOT = os.path.join("soccernet", "TrackEval", "data", "gt", "SoccerNet_tracking", "SoccerNet_tracking_2022_all_gts")
SOCCERNET_FRAMES_ROOT = os.path.join("soccernet", "soccernet_data", "tracking", "test")


def get_soccernet_dets(frame_id, dets_by_frame):
    """SoccerNet format: frame_id,-1,left,top,width,height,conf,-1,-1,-1"""
    dets = []
    for line in dets_by_frame.get(frame_id, []):
        det = line.split(",")
        x1 = int(det[2])
        y1 = int(det[3])
        x2 = int(det[4]) + int(det[2])
        y2 = int(det[5]) + int(det[3])
        dets.append([x1, y1, x2, y2, 1.0])
    return dets


tracker = BoTSORTTracker(enable_cmc=True, cmc_method="sparseOptFlow")

save_dir = os.path.join("BoTSORT_outputs_soccernet", "test_defaults_cmc")
os.makedirs(save_dir, exist_ok=True)

for seq in sorted(os.listdir(SOCCERNET_TEST_DET_ROOT)):
    tracker.reset()
    seq_name = seq.split("__")[0]

    with open(os.path.join(SOCCERNET_TEST_DET_ROOT, seq), "r") as f:
        det_list = f.readlines()
        dets_by_frame = build_dets_index(det_list)

    frames_dir = os.path.join(SOCCERNET_FRAMES_ROOT, seq_name, "img1")
    last_frame = int(det_list[-1].split(",")[0])
    output_lines = []
    for frame_id in range(1, last_frame + 1):
        raw_dets = get_soccernet_dets(frame_id, dets_by_frame)
        if raw_dets:
            raw_dets = np.array(raw_dets)
            dets = sv.Detections(xyxy=raw_dets[:, :4], confidence=raw_dets[:, 4])
        else:
            dets = sv.Detections.empty()

        frame_path = os.path.join(frames_dir, f"{frame_id:06d}.jpg")
        frame = cv2.imread(frame_path)
        if frame is None:
            frame = DUMMY_FRAME

        dets = tracker.update(detections=dets, frame=frame)
        for tid, (left, top, right, bottom) in zip(dets.tracker_id, dets.xyxy):
            if tid == -1:
                continue
            width = right - left
            height = bottom - top
            output_lines.append(
                f"{frame_id},{int(tid)},{round(left,1)},{round(top,1)},{round(width,1)},{round(height,1)},-1,-1,-1,-1\n"
            )

    with open(os.path.join(save_dir, seq_name + ".txt"), "w") as f:
        f.writelines(output_lines)
    print(f"Finished {seq_name}")

print(f"\nWrote {len(os.listdir(save_dir))} sequence files to {save_dir}")

soccernet_result = evaluate_mot_sequences(
    gt_dir=SOCCERNET_TEST_GT_ROOT,
    tracker_dir=save_dir,
    metrics=["CLEAR", "HOTA", "Identity"],
)
soccernet_agg = soccernet_result.to_dict()["aggregate"]
soccernet_metrics = {
    "dataset": "SoccerNet",
    "split": "test",
    "CMC": True,
    "HOTA": soccernet_agg["HOTA"]["HOTA"]*100,
    "IDF1": soccernet_agg["Identity"]["IDF1"]*100,
    "MOTA": soccernet_agg["CLEAR"]["MOTA"]*100,
}
print(f"SoccerNet test (CMC enabled) -> HOTA: {soccernet_metrics['HOTA']:.2f}, IDF1: {soccernet_metrics['IDF1']:.2f}, MOTA: {soccernet_metrics['MOTA']:.2f}")

Finished SNMOT-116
Finished SNMOT-117
Finished SNMOT-118
Finished SNMOT-119
Finished SNMOT-120
Finished SNMOT-121
Finished SNMOT-122
Finished SNMOT-123
Finished SNMOT-124
Finished SNMOT-125
Finished SNMOT-126
Finished SNMOT-127
Finished SNMOT-128
Finished SNMOT-129
Finished SNMOT-130
Finished SNMOT-131
Finished SNMOT-132
Finished SNMOT-133
Finished SNMOT-134
Finished SNMOT-135
Finished SNMOT-136
Finished SNMOT-137
Finished SNMOT-138
Finished SNMOT-139
Finished SNMOT-140
Finished SNMOT-141
Finished SNMOT-142
Finished SNMOT-143
Finished SNMOT-144
Finished SNMOT-145
Finished SNMOT-146
Finished SNMOT-147
Finished SNMOT-148
Finished SNMOT-149
Finished SNMOT-150
Finished SNMOT-187
Finished SNMOT-188
Finished SNMOT-189
Finished SNMOT-190
Finished SNMOT-191
Finished SNMOT-192
Finished SNMOT-193
Finished SNMOT-194
Finished SNMOT-195
Finished SNMOT-196
Finished SNMOT-197
Finished SNMOT-198
Finished SNMOT-199
Finished SNMOT-200

Wrote 49 sequence files to BoTSORT_outputs_soccernet/test_defaults_c

## SportsMOT (val set) — CMC enabled, evaluated against GT

In [8]:
SPORTSMOT_VAL_DET_ROOT = os.path.join("sportsmot", "sportsmot_yolox_dets", "val")
SPORTSMOT_VAL_FRAMES_ROOT = os.path.join("sportsmot", "val")
SPORTSMOT_GT_ROOT = os.path.join("sportsmot", "TrackEval", "data", "gt", "sportsmot")


def get_sportsmot_dets(frame_id, dets_by_frame):
    """SportsMOT format: frame_id,x1,y1,x2,y2,conf"""
    dets = []
    for line in dets_by_frame.get(frame_id, []):
        det = line.split(",")
        x1, y1, x2, y2 = float(det[1]), float(det[2]), float(det[3]), float(det[4])
        conf = float(det[5])
        dets.append([x1, y1, x2, y2, conf])
    return dets


tracker = BoTSORTTracker()

save_dir = os.path.join("BoTSORT_outputs_sportsmot", "val_defaults_cmc")
os.makedirs(save_dir, exist_ok=True)

for seq in sorted(os.listdir(SPORTSMOT_VAL_DET_ROOT)):
    if not seq.endswith(".txt"):
        continue
    tracker.reset()
    seq_name = seq.split(".")[0]

    with open(os.path.join(SPORTSMOT_VAL_DET_ROOT, seq), "r") as f:
        det_list = f.readlines()
        dets_by_frame = build_dets_index(det_list)

    frames_dir = os.path.join(SPORTSMOT_VAL_FRAMES_ROOT, seq_name, "img1")
    last_frame = int(det_list[-1].split(",")[0])
    output_lines = []
    for frame_id in range(1, last_frame + 1):
        raw_dets = get_sportsmot_dets(frame_id, dets_by_frame)
        if raw_dets:
            raw_dets = np.array(raw_dets)
            dets = sv.Detections(xyxy=raw_dets[:, :4], confidence=raw_dets[:, 4])
        else:
            dets = sv.Detections.empty()

        frame_path = os.path.join(frames_dir, f"{frame_id:06d}.jpg")
        frame = cv2.imread(frame_path)
        if frame is None:
            frame = DUMMY_FRAME

        dets = tracker.update(detections=dets, frame=frame)
        for tid, (left, top, right, bottom) in zip(dets.tracker_id, dets.xyxy):
            if tid == -1:
                continue
            width = right - left
            height = bottom - top
            output_lines.append(
                f"{frame_id},{int(tid)},{round(left,1)},{round(top,1)},{round(width,1)},{round(height,1)},-1,-1,-1,-1\n"
            )

    with open(os.path.join(save_dir, seq_name + ".txt"), "w") as f:
        f.writelines(output_lines)
    print(f"Finished {seq_name}")

print(f"\nWrote {len([f for f in os.listdir(save_dir) if f.endswith('.txt')])} sequence files to {save_dir}")

sportsmot_val_result = evaluate_mot_sequences(
    gt_dir=os.path.join(SPORTSMOT_GT_ROOT, "val"),
    tracker_dir=save_dir,
    metrics=["CLEAR", "HOTA", "Identity"],
)
sportsmot_val_agg = sportsmot_val_result.to_dict()["aggregate"]
sportsmot_val_metrics = {
    "dataset": "SportsMOT",
    "split": "val",
    "CMC": True,
    "HOTA": sportsmot_val_agg["HOTA"]["HOTA"],
    "IDF1": sportsmot_val_agg["Identity"]["IDF1"],
    "MOTA": sportsmot_val_agg["CLEAR"]["MOTA"],
}
print(f"SportsMOT val (CMC enabled) -> HOTA: {sportsmot_val_metrics['HOTA']:.3f}, IDF1: {sportsmot_val_metrics['IDF1']:.3f}, MOTA: {sportsmot_val_metrics['MOTA']:.3f}")

Finished v_00HRwkvvjtQ_c001
Finished v_00HRwkvvjtQ_c003
Finished v_00HRwkvvjtQ_c005
Finished v_00HRwkvvjtQ_c007
Finished v_00HRwkvvjtQ_c008
Finished v_00HRwkvvjtQ_c011
Finished v_0kUtTtmLaJA_c004
Finished v_0kUtTtmLaJA_c005
Finished v_0kUtTtmLaJA_c006
Finished v_0kUtTtmLaJA_c007
Finished v_0kUtTtmLaJA_c008
Finished v_0kUtTtmLaJA_c010
Finished v_2QhNRucNC7E_c017
Finished v_4-EmEtrturE_c009
Finished v_4r8QL_wglzQ_c001
Finished v_5ekaksddqrc_c001
Finished v_5ekaksddqrc_c002
Finished v_5ekaksddqrc_c003
Finished v_5ekaksddqrc_c004
Finished v_5ekaksddqrc_c005
Finished v_9MHDmAMxO5I_c002
Finished v_9MHDmAMxO5I_c003
Finished v_9MHDmAMxO5I_c004
Finished v_9MHDmAMxO5I_c006
Finished v_9MHDmAMxO5I_c009
Finished v_BgwzTUxJaeU_c008
Finished v_BgwzTUxJaeU_c012
Finished v_BgwzTUxJaeU_c014
Finished v_G-vNjfx1GGc_c004
Finished v_G-vNjfx1GGc_c008
Finished v_G-vNjfx1GGc_c600
Finished v_G-vNjfx1GGc_c601
Finished v_ITo3sCnpw_k_c007
Finished v_ITo3sCnpw_k_c010
Finished v_ITo3sCnpw_k_c011
Finished v_ITo3sCnpw

## SportsMOT (test set) — CMC enabled, submission zip

In [3]:
import shutil

SPORTSMOT_TEST_DET_ROOT = os.path.join("sportsmot", "sportsmot_yolox_dets", "test")
SPORTSMOT_FRAMES_ROOT = os.path.join("sportsmot", "test")


def get_sportsmot_dets(frame_id, dets_by_frame):
    """SportsMOT format: frame_id,x1,y1,x2,y2,conf"""
    dets = []
    for line in dets_by_frame.get(frame_id, []):
        det = line.split(",")
        x1, y1, x2, y2 = float(det[1]), float(det[2]), float(det[3]), float(det[4])
        conf = float(det[5])
        dets.append([x1, y1, x2, y2, conf])
    return dets


tracker = BoTSORTTracker(enable_cmc=True, cmc_method="sparseOptFlow")

save_dir = os.path.join("BoTSORT_outputs_sportsmot", "test_defaults_cmc")
os.makedirs(save_dir, exist_ok=True)

for seq in sorted(os.listdir(SPORTSMOT_TEST_DET_ROOT)):
    if not seq.endswith(".txt"):
        continue
    tracker.reset()
    seq_name = seq.split(".")[0]

    with open(os.path.join(SPORTSMOT_TEST_DET_ROOT, seq), "r") as f:
        det_list = f.readlines()
        dets_by_frame = build_dets_index(det_list)

    frames_dir = os.path.join(SPORTSMOT_FRAMES_ROOT, seq_name, "img1")
    last_frame = int(det_list[-1].split(",")[0])
    output_lines = []
    for frame_id in range(1, last_frame + 1):
        raw_dets = get_sportsmot_dets(frame_id, dets_by_frame)
        if raw_dets:
            raw_dets = np.array(raw_dets)
            dets = sv.Detections(xyxy=raw_dets[:, :4], confidence=raw_dets[:, 4])
        else:
            dets = sv.Detections.empty()

        frame_path = os.path.join(frames_dir, f"{frame_id:06d}.jpg")
        frame = cv2.imread(frame_path)
        if frame is None:
            frame = DUMMY_FRAME

        dets = tracker.update(detections=dets, frame=frame)
        for tid, (left, top, right, bottom) in zip(dets.tracker_id, dets.xyxy):
            if tid == -1:
                continue
            width = right - left
            height = bottom - top
            output_lines.append(
                f"{frame_id},{int(tid)},{round(left,1)},{round(top,1)},{round(width,1)},{round(height,1)},-1,-1,-1,-1\n"
            )

    with open(os.path.join(save_dir, seq_name + ".txt"), "w") as f:
        f.writelines(output_lines)
    print(f"Finished {seq_name}")

print(f"\nWrote {len([f for f in os.listdir(save_dir) if f.endswith('.txt')])} sequence files to {save_dir}")

zip_name = "BoTSORT_sportsmot_test_defaults_cmc"
shutil.make_archive(zip_name, "zip", save_dir)
print(f"Created submission: {zip_name}.zip")

sportsmot_metrics = {
    "dataset": "SportsMOT",
    "split": "test",
    "CMC": True,
    "HOTA": None,
    "IDF1": None,
    "MOTA": None,
}
print("SportsMOT test: no GT available — submit the zip to the evaluation server.")

Finished v_-9kabh1K8UA_c008
Finished v_-9kabh1K8UA_c009
Finished v_-9kabh1K8UA_c010
Finished v_-hhDbvY5aAM_c001
Finished v_-hhDbvY5aAM_c002
Finished v_-hhDbvY5aAM_c005
Finished v_-hhDbvY5aAM_c006
Finished v_-hhDbvY5aAM_c007
Finished v_-hhDbvY5aAM_c008
Finished v_-hhDbvY5aAM_c009
Finished v_-hhDbvY5aAM_c011
Finished v_-hhDbvY5aAM_c012
Finished v_-hhDbvY5aAM_c600
Finished v_-hhDbvY5aAM_c602
Finished v_1UDUODIBSsc_c001
Finished v_1UDUODIBSsc_c004
Finished v_1UDUODIBSsc_c015
Finished v_1UDUODIBSsc_c037
Finished v_1UDUODIBSsc_c055
Finished v_1UDUODIBSsc_c056
Finished v_1UDUODIBSsc_c063
Finished v_1UDUODIBSsc_c067
Finished v_1UDUODIBSsc_c090
Finished v_1UDUODIBSsc_c103
Finished v_1UDUODIBSsc_c111
Finished v_1UDUODIBSsc_c113
Finished v_1UDUODIBSsc_c601
Finished v_1UDUODIBSsc_c602
Finished v_1UDUODIBSsc_c603
Finished v_1UDUODIBSsc_c606
Finished v_1UDUODIBSsc_c609
Finished v_1UDUODIBSsc_c615
Finished v_2BhBRkkAqbQ_c002
Finished v_2ChiYdg5bxI_c039
Finished v_2ChiYdg5bxI_c044
Finished v_2ChiYdg5b

## MOT17 (test set) — CMC enabled, submission zip

In [6]:
import shutil
from pathlib import Path

MOT17_TEST_DET_ROOT = os.path.join("mot17", "MOT17_yolox_dets", "test")
MOT17_TEST_FRAMES_ROOT = os.path.join("mot17", "test")

MIN_BOX_AREA = 10
VERTICAL_RATIO_THRESH = 1.6


def get_mot17_dets(frame_id, dets_by_frame):
    """MOT17 YOLOX format: frame_id,x1,y1,x2,y2,conf"""
    dets = []
    for line in dets_by_frame.get(frame_id, []):
        det = line.split(",")
        x1, y1, x2, y2 = float(det[1]), float(det[2]), float(det[3]), float(det[4])
        conf = float(det[5])
        dets.append([x1, y1, x2, y2, conf])
    return dets


def write_mot_output(out_path: str):
    """Convert MOT17-XX.txt files into MOT17-server format."""
    out_dir = Path(out_path)
    existing = ["01", "03", "06", "07", "08", "12", "14"]
    missing = ["02", "04", "05", "09", "10", "11", "13"]
    suffixes = ["FRCNN", "SDP", "DPM"]

    for num in existing:
        src = out_dir / f"MOT17-{num}.txt"
        if not src.exists():
            print(f"  Missing expected source file: {src}")
            continue
        content = src.read_bytes()
        for suf in suffixes:
            (out_dir / f"MOT17-{num}-{suf}.txt").write_bytes(content)
        src.unlink()

    for num in missing:
        for suf in suffixes:
            (out_dir / f"MOT17-{num}-{suf}.txt").touch(exist_ok=True)

    print("  MOT17 server-format files created.")


tracker = BoTSORTTracker()

save_dir = os.path.join("BoTSORT_outputs_MOT17", "test_defaults_cmc")
os.makedirs(save_dir, exist_ok=True)

for seq in sorted(os.listdir(MOT17_TEST_DET_ROOT)):
    if not seq.endswith(".txt"):
        continue
    tracker.reset()
    seq_name = os.path.splitext(seq)[0]

    with open(os.path.join(MOT17_TEST_DET_ROOT, seq), "r") as f:
        det_list = f.readlines()
        dets_by_frame = build_dets_index(det_list)

    frames_dir = os.path.join(MOT17_TEST_FRAMES_ROOT, seq_name + "-FRCNN", "img1")
    last_frame = int(det_list[-1].split(",")[0])
    output_lines = []
    for frame_id in range(1, last_frame + 1):
        raw_dets = get_mot17_dets(frame_id, dets_by_frame)
        if raw_dets:
            raw_dets = np.array(raw_dets)
            dets = sv.Detections(xyxy=raw_dets[:, :4], confidence=raw_dets[:, 4])
        else:
            dets = sv.Detections.empty()

        frame_path = os.path.join(frames_dir, f"{frame_id:06d}.jpg")
        frame = cv2.imread(frame_path)
        if frame is None:
            frame = DUMMY_FRAME

        dets = tracker.update(detections=dets, frame=frame)
        for tid, (left, top, right, bottom) in zip(dets.tracker_id, dets.xyxy):
            if tid == -1:
                continue
            left, top, right, bottom = float(left), float(top), float(right), float(bottom)
            width = right - left
            height = bottom - top
            vertical = width / max(height, 1e-6) > VERTICAL_RATIO_THRESH
            if width * height > MIN_BOX_AREA and not vertical:
                output_lines.append(
                    f"{frame_id},{int(tid)},{round(left,1)},{round(top,1)},{round(width,1)},{round(height,1)},-1,-1,-1,-1\n"
                )

    with open(os.path.join(save_dir, seq_name + ".txt"), "w") as f:
        f.writelines(output_lines)
    print(f"Finished {seq_name}")

print(f"\nWrote {len([f for f in os.listdir(save_dir) if f.endswith('.txt')])} sequence files to {save_dir}")

write_mot_output(save_dir)

zip_name = "BoTSORT_MOT17_test_defaults_cmc"
shutil.make_archive(zip_name, "zip", save_dir)
print(f"Created submission: {zip_name}.zip")

mot17_metrics = {
    "dataset": "MOT17",
    "split": "test",
    "CMC": True,
    "HOTA": None,
    "IDF1": None,
    "MOTA": None,
}
print("MOT17 test: no GT available — submit the zip to the evaluation server.")

Finished MOT17-01
Finished MOT17-03
Finished MOT17-06
Finished MOT17-07
Finished MOT17-08
Finished MOT17-12
Finished MOT17-14

Wrote 7 sequence files to BoTSORT_outputs_MOT17/test_defaults_cmc
  MOT17 server-format files created.
Created submission: BoTSORT_MOT17_test_defaults_cmc.zip
MOT17 test: no GT available — submit the zip to the evaluation server.


## Summary

In [7]:
summary_df = pd.DataFrame([soccernet_metrics, sportsmot_val_metrics, sportsmot_metrics, mot17_metrics])
print("BoTSORT (default params) results:")
summary_df

NameError: name 'sportsmot_val_metrics' is not defined

First version

dataset	split	CMC	HOTA	IDF1	MOTA
0	SoccerNet	test	True	0.841793	0.788980	0.965146
1	DanceTrack	val	NaN	0.522681	0.510586	0.897157
2	SportsMOT	val	True	0.813284	0.805978	0.985344
3	SportsMOT	test	True	NaN	NaN	NaN
